In [3]:
"""
Class Decorators in Python
==========================

This file is a complete study note + runnable demo about class decorators.
It covers:

1. What a decorator is
2. Why a class can act as a decorator
3. How __init__ and __call__ work together
4. How invalid input data can be handled
5. Simple class decorator examples
6. Parameterized class decorators
7. Decorating methods inside classes
8. Preserving function metadata
9. Common pitfalls and edge cases

Run this file directly to see the examples.

------------------------------------------------------------
1) WHAT IS A DECORATOR?
------------------------------------------------------------

A decorator is something that takes an object, modifies or wraps it,
and returns a new object.

In Python, decorators are most often used with functions:

    @decorator_name
    def my_function(...):
        ...

This is equivalent to:

    my_function = decorator_name(my_function)

So the decorator receives the original function and returns a modified version.

------------------------------------------------------------
2) WHY USE A CLASS AS A DECORATOR?
------------------------------------------------------------

A class can act as a decorator when:

- it stores the original function in __init__
- it makes its objects callable using __call__

That means the class instance can behave like a function.

This is useful when you want:

- state inside the decorator
- configurable behavior
- reusable validation / logging / timing logic
- more complex behavior than a simple function decorator

------------------------------------------------------------
3) BASIC CLASS DECORATOR
------------------------------------------------------------

The simplest structure is:

    class Decorator:
        def __init__(self, func):
            self.func = func

        def __call__(self, *args, **kwargs):
            return self.func(*args, **kwargs)

When you write:

    @Decorator
    def add(a, b):
        return a + b

Python converts it to:

    add = Decorator(add)

Now `add` is no longer the original function.
It is an instance of `Decorator`.
When you call `add(2, 3)`, Python actually runs:

    Decorator_instance.__call__(2, 3)

------------------------------------------------------------
4) HOW INVALID INPUT DATA IS HANDLED
------------------------------------------------------------

A class decorator can validate input before calling the function.
For example:

- check whether arguments are integers
- check for division by zero
- check for allowed range
- check for empty input

If input is invalid, it can:

- raise TypeError / ValueError
- return a custom message
- log the problem
- fall back to a default value

Best practice:
- raise an exception for programmer errors
- let the caller handle it using try/except if needed

------------------------------------------------------------
5) WHY __call__ IS IMPORTANT
------------------------------------------------------------

__call__ allows an object to be called like a function.

Example:

    obj = SomeClass()
    obj()

This works only if __call__ exists.

For decorators, this is crucial because the decorated object should still behave like a function.
Without __call__, this would fail.

------------------------------------------------------------
6) A VERY SIMPLE EXAMPLE
------------------------------------------------------------
"""


'\nClass Decorators in Python\n==========================\n\nThis file is a complete study note + runnable demo about class decorators.\nIt covers:\n\n1. What a decorator is\n2. Why a class can act as a decorator\n3. How __init__ and __call__ work together\n4. How invalid input data can be handled\n5. Simple class decorator examples\n6. Parameterized class decorators\n7. Decorating methods inside classes\n8. Preserving function metadata\n9. Common pitfalls and edge cases\n\nRun this file directly to see the examples.\n\n------------------------------------------------------------\n1) WHAT IS A DECORATOR?\n------------------------------------------------------------\n\nA decorator is something that takes an object, modifies or wraps it,\nand returns a new object.\n\nIn Python, decorators are most often used with functions:\n\n    @decorator_name\n    def my_function(...):\n        ...\n\nThis is equivalent to:\n\n    my_function = decorator_name(my_function)\n\nSo the decorator receives

In [4]:

from __future__ import annotations

from functools import update_wrapper
from time import perf_counter
from typing import Any, Callable


# ============================================================
# EXAMPLE 1: SIMPLE CLASS DECORATOR
# ============================================================

class SimpleLogger:
    """A basic class decorator that logs calls and returns the original result."""

    def __init__(self, func: Callable[..., Any]):
        self.func = func
        update_wrapper(self, func)  # preserves __name__, __doc__, etc.

    def __call__(self, *args: Any, **kwargs: Any) -> Any:
        print(f"[SimpleLogger] Calling {self.func.__name__} with args={args}, kwargs={kwargs}")
        result = self.func(*args, **kwargs)
        print(f"[SimpleLogger] {self.func.__name__} returned {result}")
        return result


@SimpleLogger
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


# ============================================================
# EXAMPLE 2: CLASS DECORATOR THAT MODIFIES THE RESULT
# ============================================================

class DoubleResult:
    """Calls the original function and doubles the returned value."""

    def __init__(self, func: Callable[..., Any]):
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args: Any, **kwargs: Any) -> Any:
        result = self.func(*args, **kwargs)
        return result * 2


@DoubleResult
def add_and_return(a: int, b: int) -> int:
    return a + b


# ============================================================
# EXAMPLE 3: INPUT VALIDATION WITH CLASS DECORATOR
# ============================================================

class ValidateIntegers:
    """Validate that all positional arguments are integers.

    If any argument is not an int, raise TypeError.
    This is the kind of decorator that answers the question:
    "How do class decorators handle invalid input data?"
    """

    def __init__(self, func: Callable[..., Any]):
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args: Any, **kwargs: Any) -> Any:
        # Validate positional arguments
        for index, value in enumerate(args):
            if not isinstance(value, int):
                raise TypeError(
                    f"{self.func.__name__} expects only integers. "
                    f"Argument at position {index} is {value!r} ({type(value).__name__})."
                )

        # Validate keyword arguments too
        for key, value in kwargs.items():
            if not isinstance(value, int):
                raise TypeError(
                    f"{self.func.__name__} expects only integers. "
                    f"Keyword argument '{key}' is {value!r} ({type(value).__name__})."
                )

        return self.func(*args, **kwargs)


@ValidateIntegers
def safe_add(a: int, b: int) -> int:
    return a + b


# ============================================================
# EXAMPLE 4: PARAMETERIZED CLASS DECORATOR
# ============================================================

class Repeat:
    """A class decorator with configuration.

    Usage:
        @Repeat(times=3)
        def hello():
            print("Hi")

    This is NOT the same as @Repeat directly.
    Here Repeat is first initialized with configuration,
    and then the returned object is used as the actual decorator.
    """

    def __init__(self, times: int = 2):
        self.times = times

    def __call__(self, func: Callable[..., Any]):
        update_wrapper(self, func)

        def wrapper(*args: Any, **kwargs: Any):
            result = None
            for i in range(self.times):
                print(f"[Repeat] Run {i + 1}/{self.times}")
                result = func(*args, **kwargs)
            return result

        update_wrapper(wrapper, func)
        return wrapper


@Repeat(times=3)
def greet(name: str) -> None:
    print(f"Hello, {name}!")


# ============================================================
# EXAMPLE 5: TIMING DECORATOR USING A CLASS
# ============================================================

class Timer:
    """Measure execution time of the wrapped function."""

    def __init__(self, func: Callable[..., Any]):
        self.func = func
        update_wrapper(self, func)

    def __call__(self, *args: Any, **kwargs: Any) -> Any:
        start = perf_counter()
        try:
            return self.func(*args, **kwargs)
        finally:
            end = perf_counter()
            print(f"[Timer] {self.func.__name__} took {end - start:.6f} seconds")


@Timer
def slow_square(n: int) -> int:
    total = 0
    for _ in range(1_000_00):
        total += n * n
    return total


# ============================================================
# EXAMPLE 6: DECORATING METHODS INSIDE A CLASS
# ============================================================

class MethodAwareDecorator:
    """A class decorator that also works as a descriptor.

    This is important for methods defined inside classes.
    Without __get__, method binding can break.

    When used on a normal function:
        @MethodAwareDecorator
        def func(...):
            ...

    When used on a method inside a class, Python should still be able
    to bind self correctly. __get__ helps with that.
    """

    def __init__(self, func: Callable[..., Any]):
        self.func = func
        update_wrapper(self, func)

    def __get__(self, instance, owner):
        # Descriptor protocol: return a bound callable when accessed via instance.
        if instance is None:
            return self

        def bound(*args: Any, **kwargs: Any):
            return self.__call__(instance, *args, **kwargs)

        update_wrapper(bound, self.func)
        return bound

    def __call__(self, *args: Any, **kwargs: Any) -> Any:
        print(f"[MethodAwareDecorator] Calling {self.func.__name__}")
        return self.func(*args, **kwargs)


class Calculator:
    @MethodAwareDecorator
    def multiply(self, a: int, b: int) -> int:
        return a * b



# ============================================================
# EXAMPLE 7: HANDLING INVALID INPUT GRACEFULLY WITH TRY/EXCEPT
# ============================================================

@ValidateIntegers
def divide(a: int, b: int) -> float:
    if b == 0:
        raise ZeroDivisionError("division by zero is not allowed")
    return a / b



In [5]:

# ============================================================
# COMMON EDGE CASES / NOTES
# ============================================================

"""
Edge case 1: Function metadata
--------------------------------

Without update_wrapper, the decorated object may lose:
- __name__
- __doc__
- __module__

update_wrapper helps preserve those details.

Edge case 2: Returning a new function vs returning self
--------------------------------------------------------

A class decorator can:
- return self from __init__/__call__ style setup
- or return an inner wrapper function from __call__

Both are valid. Which one you choose depends on whether you need state.

Edge case 3: Methods inside classes
-----------------------------------

When a decorator wraps a method, the first argument is usually self.
The decorator must not accidentally remove the binding behavior.
That is why descriptor support (__get__) can be useful.

Edge case 4: Decorating functions with variable arguments
---------------------------------------------------------

Use *args and **kwargs in __call__ so the decorator can work with
almost any function signature.

Edge case 5: Reusing the same decorator object
----------------------------------------------

If the decorator stores mutable state, the same instance may be reused
in surprising ways. Be careful with counters, caches, or shared lists.

Edge case 6: Exception handling
-------------------------------

A decorator may catch exceptions, log them, convert them, or re-raise them.
For invalid input, re-raising TypeError / ValueError is usually best.

"""



'\nEdge case 1: Function metadata\n--------------------------------\n\nWithout update_wrapper, the decorated object may lose:\n- __name__\n- __doc__\n- __module__\n\nupdate_wrapper helps preserve those details.\n\nEdge case 2: Returning a new function vs returning self\n--------------------------------------------------------\n\nA class decorator can:\n- return self from __init__/__call__ style setup\n- or return an inner wrapper function from __call__\n\nBoth are valid. Which one you choose depends on whether you need state.\n\nEdge case 3: Methods inside classes\n-----------------------------------\n\nWhen a decorator wraps a method, the first argument is usually self.\nThe decorator must not accidentally remove the binding behavior.\nThat is why descriptor support (__get__) can be useful.\n\nEdge case 4: Decorating functions with variable arguments\n---------------------------------------------------------\n\nUse *args and **kwargs in __call__ so the decorator can work with\nalmost 

In [6]:

# ============================================================
# DEMONSTRATION CODE
# ============================================================

if __name__ == "__main__":
    print("\n========== BASIC CLASS DECORATOR ==========")
    print("add(2, 3) =>", add(2, 3))

    print("\n========== RESULT MODIFICATION ==========")
    print("add_and_return(2, 3) =>", add_and_return(2, 3))

    print("\n========== INPUT VALIDATION ==========")
    try:
        print("safe_add(10, 20) =>", safe_add(10, 20))
        print("safe_add(10, '20') =>", safe_add(10, '20'))
    except TypeError as e:
        print("Caught TypeError:", e)

    print("\n========== PARAMETERIZED DECORATOR ==========")
    greet("Rajan")

    print("\n========== TIMER DECORATOR ==========")
    print("slow_square(5) =>", slow_square(5))

    print("\n========== DECORATING METHODS ==========")
    calc = Calculator()
    print("calc.multiply(4, 5) =>", calc.multiply(4, 5))

    print("\n========== DIVISION WITH VALIDATION ==========")
    try:
        print("divide(10, 2) =>", divide(10, 2))
        print("divide(10, 0) =>", divide(10, 0))
    except Exception as e:
        print("Caught exception:", type(e).__name__, e)




========== BASIC CLASS DECORATOR ==========
[SimpleLogger] Calling add with args=(2, 3), kwargs={}
[SimpleLogger] add returned 5
add(2, 3) => 5

========== RESULT MODIFICATION ==========
add_and_return(2, 3) => 10

========== INPUT VALIDATION ==========
safe_add(10, 20) => 30
Caught TypeError: safe_add expects only integers. Argument at position 1 is '20' (str).

========== PARAMETERIZED DECORATOR ==========
[Repeat] Run 1/3
Hello, Rajan!
[Repeat] Run 2/3
Hello, Rajan!
[Repeat] Run 3/3
Hello, Rajan!

========== TIMER DECORATOR ==========
[Timer] slow_square took 0.011687 seconds
slow_square(5) => 2500000

========== DECORATING METHODS ==========
[MethodAwareDecorator] Calling multiply
calc.multiply(4, 5) => 20

========== DIVISION WITH VALIDATION ==========
divide(10, 2) => 5.0
Caught exception: ZeroDivisionError division by zero is not allowed


In [7]:

# ============================================================
# DIRECT ANSWERS TO THE TWO QUESTIONS
# ============================================================

"""
Q1) How do class decorators handle invalid input data?

A class decorator can check the inputs inside __call__ before the wrapped
function runs.

Typical process:

1. Receive the arguments in __call__(*args, **kwargs)
2. Validate them using isinstance, range checks, emptiness checks, etc.
3. If invalid:
   - raise TypeError / ValueError
   - or return a custom error message
   - or log and stop execution
4. If valid:
   - call the original function
   - return its result

Example:

    @ValidateIntegers
    def add(a, b):
        return a + b

    add(1, 2)      # works
    add(1, '2')    # TypeError before the function runs

So the decorator acts like a guard in front of the function.

Q2) Why use the magic __call__ method in decorators?

Because __call__ makes an object callable like a function.

When a class instance has __call__, you can do:

    obj(...)

That is exactly what we need for a class decorator.

Without __call__, this would only be a normal object and not a callable wrapper.
With __call__, the decorated object can:

- accept arguments
- run validation
- call the original function
- modify the result
- return the final output

In short:
- __init__ stores the function
- __call__ runs every time the decorated function is called

That is the heart of class-based decorators.
"""


"\nQ1) How do class decorators handle invalid input data?\n\nA class decorator can check the inputs inside __call__ before the wrapped\nfunction runs.\n\nTypical process:\n\n1. Receive the arguments in __call__(*args, **kwargs)\n2. Validate them using isinstance, range checks, emptiness checks, etc.\n3. If invalid:\n   - raise TypeError / ValueError\n   - or return a custom error message\n   - or log and stop execution\n4. If valid:\n   - call the original function\n   - return its result\n\nExample:\n\n    @ValidateIntegers\n    def add(a, b):\n        return a + b\n\n    add(1, 2)      # works\n    add(1, '2')    # TypeError before the function runs\n\nSo the decorator acts like a guard in front of the function.\n\nQ2) Why use the magic __call__ method in decorators?\n\nBecause __call__ makes an object callable like a function.\n\nWhen a class instance has __call__, you can do:\n\n    obj(...)\n\nThat is exactly what we need for a class decorator.\n\nWithout __call__, this would only